# Population distribution as 3D columns

This additional case study uses **basemap3d**, an unofficial Python wrapper for basemap.de, in a second city. Package explanations are English; the Berlin case-study instructions and viewer are German. The data is a small, reproducible extract of the official Zensus 2022 population grid, not synthetic population.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/Basemap3D/blob/main/notebooks/Population_3D_Columns.ipynb)

**Colab:** click the badge and run the setup cell. It prepares the package and the bundled CSV automatically; no desktop GIS or local Python installation is needed. Local Jupyter users follow `README.md`; the setup cell switches to the repository root when needed.

The API remains Folium-like: create `Map`, add `BarLayer(...).add_to(m)`, display `m`, and export with `m.save(...)`. Data stays in a CSV file, not in notebook code.
**Colab:** Eine verbundene Laufzeit ist erforderlich. `m` bzw. `p2` am Zellenende (alternativ `.show()`) zeigt die Karte über den automatisch verwalteten Colab-Webserver an. Go Live und manuelle Serverbefehle sind nicht nötig. Nach einem Update eine neue Laufzeit verwenden und alle Zellen erneut ausführen.


In [ ]:
import os, subprocess, sys
if "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules:
    target = "/content/basemap3d-course"
    if not os.path.isdir(target + "/.git"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/yfeng-hsm/Basemap3D.git", target], check=True)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    sys.path.insert(0, target)
elif os.path.isdir("../basemap3d"):
    os.chdir("..")


In [ ]:
from basemap3d import Map, BarLayer

## 1 · Prepare the geographical context

### Fallbeispiel Berlin: Bevölkerung im 1-km-Gitter

**Lernziel:** Erkennen Sie die räumliche Lage des Untersuchungsgebiets, bevor ein statistischer Wert dreidimensional dargestellt wird. Für diese thematische Karte werden die LoD2-Gebäude deaktiviert, damit echte Gebäude nicht mit Bevölkerungssäulen verwechselt werden.

Die Daten beziehen sich auf den **15.05.2022**. Enthalten sind 657 veröffentlichte, bewohnte 1-km-Zellen, deren Mittelpunkte in einem rechteckigen Ausschnitt im Raum Berlin liegen. Das Rechteck ist **nicht** die Berliner Verwaltungsgrenze. Fehlende Zellen sind keine gemessenen Nullwerte.

In [ ]:
p1 = Map(location=[52.51, 13.39], zoom_start=12, distance_m=34000,
         buildings=False, heading=0, pitch=-65,
         title="Zensus 2022 · Bevölkerung im Raum Berlin")
p1.save("results/population_context.html")
p1

## 2 · Add the population columns

### Bevölkerung als Säulenkartogramm

Jede Säule steht am Mittelpunkt einer ursprünglichen 1-km-Gitterzelle. **Höhe und Farbe kodieren die veröffentlichte Bevölkerungszahl dieser Zelle.** Die 800 m breite Säule ist ein Symbol; sie ist weder der echte Zellumriss noch die Wohnfläche der Personen.

Der Höhenfaktor 0,06 bedeutet: 1.000 Personen werden als 60 m hohe Säule dargestellt. Das ist eine grafische Skalierung und keine Gebäudehöhe. Die fünf Farbklassen beruhen auf Quantilen des gesamten geladenen Datensatzes und bleiben beim Filtern unverändert.

In [ ]:
p2 = p1.copy()
BarLayer("data/berlin_population_1km.csv", value="population",
         height_scale=0.06, bar_width_m=800,
         attribution="© Statistische Ämter des Bundes und der Länder, 2024 · Zensus 2022 · dl-de/by-2-0").add_to(p2)
p2.save("results/population_berlin.html")
p2

## 3 · Explore and explain the map

### Aufgaben

1. Drehen Sie die Ansicht. Wo sind höhere oder niedrigere Werte räumlich konzentriert?
2. Ändern Sie den **Höhenfaktor**. Was verändert sich an der Wirkung der Karte, obwohl die Daten gleich bleiben?
3. Erhöhen Sie den **Mindestwert**. Welche Zellen bleiben sichtbar? Die Zahlen und Farbgrenzen werden dabei nicht neu berechnet.
4. Klicken Sie auf eine Säule. Prüfen Sie Gitter-ID, Bevölkerungszahl und Stichtag im Popup.
5. Erklären Sie, warum die Summe dieses Ausschnitts keine amtliche Einwohnerzahl Berlins ist und warum keine einzelnen Wohnadressen ablesbar sind.

**Abgabe:** Notebook, exportierte HTML-Datei und ein Screenshot mit Quellenangabe. Beschreiben Sie zwei räumliche Muster und eine Grenze dieser Darstellung.

### Provenance and limitations

- Source: [Statistische Ämter des Bundes und der Länder / Destatis](https://www.destatis.de/DE/Themen/Gesellschaft-Umwelt/Bevoelkerung/Zensus2022/_publikationen.html), Zensus 2022, `Zensus2022_Bevoelkerungszahl_1km-Gitter.csv`; reference date 2022-05-15. Licence: dl-de/by-2-0.
- The source's EPSG:3035 cell centres were converted to WGS84. Published counts were not interpolated, simulated or altered. The rectangular subset and projection process are recorded in `data/berlin_population_1km.metadata.json`; `tools/prepare_population.py` reproduces the extract from the official archive.
- Cell-Key confidentiality processing perturbs some counts. The data is a historical census snapshot, not current population or real-time occupancy. An absent grid cell is not necessarily zero.
- Each symbol represents an equal-area 1-km source cell. Population is not divided by the narrower display-column footprint. The 3D height is a cartographic encoding, not a physical height.
- The saved HTML embeds the small CSV-derived dataset but loads map/terrain online. In Jupyter or Colab, the last expression `p2` displays the scene.